In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import torch.utils.data as data
from monai.data import DataLoader, ImageDataset
import torch.nn as nn
import optuna
import torch
from sklearn.model_selection import StratifiedKFold
from monai.networks.nets import EfficientNetBN, DenseNet121
import seaborn as sns
import monai
from sklearn import metrics
import nibabel as nib
from train import *
from val import *
from image_dataset import *
import json
from imblearn.over_sampling import RandomOverSampler
from sklearn.model_selection import train_test_split
from torch.utils.data import WeightedRandomSampler

In [3]:
dev_file = "/research/projects/Nakai/clf_artifacts/T2combined_datasets_new/dev.xlsx"
test_file = "/research/projects/Nakai/clf_artifacts/T2combined_datasets_new/test.xlsx"
dev = pd.read_excel(dev_file)
test = pd.read_excel(test_file)

In [4]:
def label_encode(label):
    """convert label for ordinal classification, e.g.
    0 -> [1,0,0,0]
    1 -> [1,1,0,0]
    2 -> [1,1,1,0]
    https://towardsdatascience.com/how-to-perform-ordinal-regression-classification-in-pytorch-361a2a095a99
    """
    ordinal_label = {0:np.array([1,0,0,0]), 1:np.array([1,1,0,0]), 2:np.array([1,1,1,0]), 3:np.array([1,1,1,1])}[label]
    return ordinal_label

In [5]:
def main(train, val, settings, fold=""):
    batch_size = settings["batch_size"] #4
    epochs = settings["epochs"] #35
    settings["learning_rate_Tmax"] = epochs
    device = settings["device"] #"cuda"
    clf = settings["clf"] # 'ordinal_classification'
    num_classes = settings["num_classes"] #4
    spatial_size = settings["spatial_size"] #(224, 224, -1)
    center_crop_xy = settings["center_crop_xy"] #-1
    z = settings["z"] #16
    roi_size = (-1, -1, z)
    use_mask = settings["use_mask"] #1
    image_path_col = settings["image_path_col"]
    pin_memory = settings["pin_memory"]
    mask_path_col = image_path_col.replace("image", "mask")
    
    # Create Dataloader
    train_transform = create_transform_T2(spatial_size=spatial_size, roi_size=roi_size, set="train", use_mask=use_mask,
                                          center_crop_xy=center_crop_xy)
    val_transform = create_transform_T2(spatial_size=spatial_size, roi_size=roi_size, set="val", use_mask=use_mask,
                                        center_crop_xy=center_crop_xy)
    
    train_img = train[image_path_col].values
    train_mask = train[mask_path_col].values
    train_categorical_label = train["final_score"].values
    
    # Validation dataset
    val_img = val[image_path_col].values
    val_mask = val[mask_path_col].values
    val_categorical_label = val["final_score"].values
    
    class_sample_count = np.array(
        [len(np.where(train_categorical_label == t)[0]) for t in np.unique(train_categorical_label)])
    weight = 1. / class_sample_count
    samples_weight = np.array([weight[t] for t in train_categorical_label])
    samples_weight = torch.from_numpy(samples_weight)
    sampler = WeightedRandomSampler(samples_weight, len(samples_weight))
    
    # convert labels for ordinal classification
    train_label = np.array(pd.Series(train_categorical_label).apply(label_encode).to_list()).astype(np.float32)
    val_label = np.array(pd.Series(val_categorical_label).apply(label_encode).to_list()).astype(np.float32)

    # Train five models within the development set
    train_dataset = ProstateDataset(img_path=train_img, mask_path=train_mask, label=train_label, transform=train_transform)
    val_dataset = ProstateDataset(img_path=val_img, mask_path=val_mask, label=val_label, transform=val_transform)
    
    # Dataloader 
    train_loader = DataLoader(train_dataset, batch_size=batch_size, num_workers=4,
                              #shuffle=True, 
                              pin_memory=pin_memory, sampler=sampler)  ##### True ######
    val_loader = DataLoader(val_dataset, batch_size=batch_size, num_workers=4, shuffle=False, pin_memory=pin_memory)
    
    # 
    in_channels = 1 + int(use_mask)
    model = EfficientNetBN("efficientnet-b0", spatial_dims=3, in_channels=in_channels, num_classes=num_classes).to(device)
    
    path_to_save_model = Path(f"/research/projects/Nakai/clf_artifacts/models/T2_v3_{clf}_fold{fold}.pt")
    
    # model development
    train_losses, val_losses = model_training(model, train_loader, val_loader, params=settings, epochs=epochs,
                                              device=device, clf=clf, path_to_save_model=path_to_save_model)

    return train_loader, val_loader, path_to_save_model

In [ ]:
setting_file = "settings_T2_v3.xlsx"
settings_df = pd.read_excel(setting_file)

# str -> tuple
settings_df["spatial_size"] = settings_df["spatial_size"].apply(eval)

In [8]:
# Train models using 5-fold cross validation
for fold in range(5): #5
    print(f"Fold {fold}")
    train = dev[dev["fold"]!=fold]
    val = dev[dev["fold"]==fold]
    # Model training
    train_loader, val_loader, path_to_save_model = main(train=train, val=val, settings=settings, fold=fold)
    torch.cuda.empty_cache()

Fold 0


/home/m276992/anaconda3/lib/python3.11/site-packages/monai/utils/deprecate_utils.py:321: FutureWarning: image_dataset CropForeground_CustomFOVsized.__init__:allow_smaller: Current default value of argument `allow_smaller=True` has been deprecated since version 1.2. It will be changed to `allow_smaller=False` in version 1.5.
  warn_deprecated(argname, msg, warning_category)


Start training.. {'batch_size': 4, 'epochs': 35, 'device': 'cuda', 'clf': 'ordinal_classification', 'num_classes': 4, 'spatial_size': (224, 224, -1), 'center_crop_xy': -1, 'z': 16, 'use_mask': 1, 'loss_function_weight': None, 'over_sample': 1, 'learning_rate': 0.005, 'image_path_col': 'non_crop_image_path', 'pin_memory': 1, 'fold0_val_auc': nan, 'fold1_val_auc': nan, 'fold2_val_auc': nan, 'fold3_val_auc': 0.938749312305153, 'fold4_val_auc': 0.892564745196324, 'learning_rate_Tmax': 35}
0, 11.82659790604267, 0.28219491156727766
lr: 0.0049901370202883365
1, 0.5674277572394457, 2.5944600826086
lr: 0.004960627492066643
2, 0.3345371899303309, 0.3241437134701152
lr: 0.004911709008704841
3, 0.25676689857920243, 1.4286544433426718
lr: 0.004843775433067357
4, 0.2253760350218346, 0.1440959632136794
lr: 0.004757373726360928
5, 0.19636279303407253, 0.3363693260349507
lr: 0.004653199544324573
6, 0.1891478796402902, 0.13800496291802372
lr: 0.004532091636218622
7, 0.19080181885510683, 0.14287770877397

/home/m276992/anaconda3/lib/python3.11/site-packages/monai/utils/deprecate_utils.py:321: FutureWarning: image_dataset CropForeground_CustomFOVsized.__init__:allow_smaller: Current default value of argument `allow_smaller=True` has been deprecated since version 1.2. It will be changed to `allow_smaller=False` in version 1.5.
  warn_deprecated(argname, msg, warning_category)


Start training.. {'batch_size': 4, 'epochs': 35, 'device': 'cuda', 'clf': 'ordinal_classification', 'num_classes': 4, 'spatial_size': (224, 224, -1), 'center_crop_xy': -1, 'z': 16, 'use_mask': 1, 'loss_function_weight': None, 'over_sample': 1, 'learning_rate': 0.005, 'image_path_col': 'non_crop_image_path', 'pin_memory': 1, 'fold0_val_auc': nan, 'fold1_val_auc': nan, 'fold2_val_auc': nan, 'fold3_val_auc': 0.938749312305153, 'fold4_val_auc': 0.892564745196324, 'learning_rate_Tmax': 35}
0, 13.415637809176777, 0.407623483344566
lr: 0.0049901370202883365
1, 0.658178410384544, 0.25550867287918577
lr: 0.004960627492066643
2, 0.45579523164345775, 0.3251451948700949
lr: 0.004911709008704841
3, 0.32727099822964084, 0.1787001547771831
lr: 0.004843775433067357
4, 0.24626724850819554, 0.16086659051997718
lr: 0.004757373726360928
5, 0.20562091186036205, 0.19503104288217632
lr: 0.004653199544324573
6, 0.19505079300684292, 0.1975918471986471
lr: 0.004532091636218622
7, 0.18102727763268145, 0.17540973

/home/m276992/anaconda3/lib/python3.11/site-packages/monai/utils/deprecate_utils.py:321: FutureWarning: image_dataset CropForeground_CustomFOVsized.__init__:allow_smaller: Current default value of argument `allow_smaller=True` has been deprecated since version 1.2. It will be changed to `allow_smaller=False` in version 1.5.
  warn_deprecated(argname, msg, warning_category)


Start training.. {'batch_size': 4, 'epochs': 35, 'device': 'cuda', 'clf': 'ordinal_classification', 'num_classes': 4, 'spatial_size': (224, 224, -1), 'center_crop_xy': -1, 'z': 16, 'use_mask': 1, 'loss_function_weight': None, 'over_sample': 1, 'learning_rate': 0.005, 'image_path_col': 'non_crop_image_path', 'pin_memory': 1, 'fold0_val_auc': nan, 'fold1_val_auc': nan, 'fold2_val_auc': nan, 'fold3_val_auc': 0.938749312305153, 'fold4_val_auc': 0.892564745196324, 'learning_rate_Tmax': 35}
0, 5.95172130281842, 0.176559304948463
lr: 0.0049901370202883365
1, 0.4532399003415607, 0.27989310563303704
lr: 0.004960627492066643
2, 0.24478097317393782, 0.17731221747952838
lr: 0.004911709008704841
3, 0.21591338175240643, 0.21180758668586266
lr: 0.004843775433067357
4, 0.20772672840935547, 0.15759329792372015
lr: 0.004757373726360928
5, 0.19027800177938717, 0.18498447589402975
lr: 0.004653199544324573
6, 0.1840736682981599, 0.203284130540005
lr: 0.004532091636218622
7, 0.17810840772594824, 0.178758217

/home/m276992/anaconda3/lib/python3.11/site-packages/monai/utils/deprecate_utils.py:321: FutureWarning: image_dataset CropForeground_CustomFOVsized.__init__:allow_smaller: Current default value of argument `allow_smaller=True` has been deprecated since version 1.2. It will be changed to `allow_smaller=False` in version 1.5.
  warn_deprecated(argname, msg, warning_category)


0, 10.651225993626339, 0.1963340061348538
lr: 0.0049901370202883365
1, 0.6259364454768771, 0.25916359019140867
lr: 0.004960627492066643
2, 0.27493832821329667, 0.1897037307796783
lr: 0.004911709008704841
3, 0.22357638788864362, 0.3004304720738599
lr: 0.004843775433067357
4, 0.20840475649767837, 0.18651959577272104
lr: 0.004757373726360928
5, 0.1966695869774666, 0.1732199066426865
lr: 0.004653199544324573
6, 0.18793894142605538, 0.16089271935959196
lr: 0.004532091636218622
7, 0.18264656875629065, 0.16551217670704044
lr: 0.004395025091708848
8, 0.17507314187106351, 0.192710044252318
lr: 0.0042431034900178195
9, 0.1707334634847939, 0.16639962333232858
lr: 0.004077550014553898
10, 0.16782989218657793, 0.14012425557471986
lr: 0.0038996976045576515
11, 0.17359286561868217, 0.16054818363383758
lr: 0.003710978223058847
12, 0.1723352814158208, 0.19071871527405673
lr: 0.0035129113275521135
13, 0.16504960201767294, 0.14489890981552211
lr: 0.0033070916362186215
14, 0.1654875482389227, 0.1424880412

KeyboardInterrupt: 

In [14]:
for fold in range(3,4): #5
    print(f"Fold {fold}")
    train = dev[dev["fold"]!=fold]
    val = dev[dev["fold"]==fold]
    # Train model using different hyperparams
    settings_df_index = 0
    settings = settings_df.loc[settings_df_index].to_dict()
    settings["loss_function_weight"] = None
    # Model training
    train_loader, val_loader, path_to_save_model = main(train=train, val=val, settings=settings, fold=fold)
    torch.cuda.empty_cache()

Fold 3


/home/m276992/anaconda3/lib/python3.11/site-packages/monai/utils/deprecate_utils.py:321: FutureWarning: image_dataset CropForeground_CustomFOVsized.__init__:allow_smaller: Current default value of argument `allow_smaller=True` has been deprecated since version 1.2. It will be changed to `allow_smaller=False` in version 1.5.
  warn_deprecated(argname, msg, warning_category)


Start training.. {'batch_size': 4, 'epochs': 35, 'device': 'cuda', 'clf': 'ordinal_classification', 'num_classes': 4, 'spatial_size': (224, 224, -1), 'center_crop_xy': -1, 'z': 16, 'use_mask': 1, 'loss_function_weight': None, 'over_sample': 1, 'learning_rate': 0.005, 'image_path_col': 'non_crop_image_path', 'pin_memory': 1, 'fold0_val_auc': nan, 'fold1_val_auc': nan, 'fold2_val_auc': nan, 'fold3_val_auc': 0.938749312305153, 'fold4_val_auc': 0.892564745196324, 'learning_rate_Tmax': 35}
0, 6.999671692678402, 0.27695466474045155
lr: 0.0049901370202883365
1, 0.3311614088800757, 0.32716841519225476
lr: 0.004960627492066643
2, 0.23866992961442055, 0.32913355494654456
lr: 0.004911709008704841
3, 0.2083873801193265, 0.3414676380885202
lr: 0.004843775433067357
4, 0.19680024879486407, 0.21109569696492927
lr: 0.004757373726360928
5, 0.19245065069692427, 0.3181076285450958
lr: 0.004653199544324573
6, 0.1827030772733134, 0.2614877461347469
lr: 0.004532091636218622
7, 0.18538339366746504, 0.23727132